<a href="https://colab.research.google.com/github/divyadharshini-1306/Mutual-Fund-Style-Drifter/blob/main/style_drift_detector_data_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mutual Fund Style Drift Detector

**Project:** Detect when a mutual fund manager shifts holdings away from the fund's declared category using K-Means clustering and PCA on AMFI monthly portfolio disclosures.

**Dataset:** AMFI monthly portfolio disclosures (Jul 2024 – Jun 2025) · AMFI cap classification lists · NSE benchmark constituents

**Techniques:** K-Means clustering · PCA · Anomaly detection · Time series drift tracking

**Author:** Divyadharshini | PES University
**GitHub:** github.com/divyadharshini-1306

In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully.


In [2]:
import os

# Base path — every file in this project lives under here
BASE = "/content/drive/MyDrive/StyleDrift_Project"

# All folders we need
folders = [
    f"{BASE}/raw_data/amfi_portfolio",   # Dataset 1: monthly Excel files
    f"{BASE}/raw_data/amfi_cap_list",     # Dataset 2: cap classification lists
    f"{BASE}/raw_data/nse_benchmarks",    # Dataset 3: Nifty index constituents
    f"{BASE}/processed",                  # Cleaned DataFrames saved as CSV
    f"{BASE}/outputs",                    # Charts, results, flagged fund lists
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)   # exist_ok=True means no error if folder already exists
    print(f"  {folder}")

print("\nAll folders ready.")

  /content/drive/MyDrive/StyleDrift_Project/raw_data/amfi_portfolio
  /content/drive/MyDrive/StyleDrift_Project/raw_data/amfi_cap_list
  /content/drive/MyDrive/StyleDrift_Project/raw_data/nse_benchmarks
  /content/drive/MyDrive/StyleDrift_Project/processed
  /content/drive/MyDrive/StyleDrift_Project/outputs

All folders ready.


In [3]:
# Walk the folder tree and print it so you can confirm everything is in place

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "    " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

StyleDrift_Project/
    raw_data/
        amfi_portfolio/
        amfi_cap_list/
            Average_Market_Capitalization_30_Jun2024_2a1ab4c1d8.xlsx
            AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2024.xlsx
            AverageMarketCapitalization31Dec2025.xlsx
            AverageMarketCapitalization30Jun2025.xlsx
        nse_benchmarks/
    processed/
        holdings_master.csv
    outputs/


In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://www.advisorkhoj.com/form-download-centre/Mutual/SBI-Mutual-Fund"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers, timeout=15)

print("Status code:", resp.status_code)
print("Page length:", len(resp.text))
print("Contains 'Monthly Portfolio Disclosure'?", "Monthly Portfolio Disclosure" in resp.text)
print("Contains '.xlsx'?", ".xlsx" in resp.text)

Status code: 200
Page length: 170338
Contains 'Monthly Portfolio Disclosure'? True
Contains '.xlsx'? True


In [5]:
import requests
from bs4 import BeautifulSoup

url = "https://www.advisorkhoj.com/form-download-centre/Mutual/SBI-Mutual-Fund"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers, timeout=15)
soup = BeautifulSoup(resp.text, "html.parser")

links = {}
for a in soup.find_all("a", href=True):
    text = a.get_text(strip=True)
    href = a["href"]
    if "Monthly Portfolio Disclosure" in text and ".xlsx" in href:
        links[text] = href

print(len(links), "files found")
for label, link in list(links.items())[:5]:
    print(label, "->", link)

41 files found
Monthly Portfolio Disclosure - May 2026 -> https://www.sbimf.com/docs/default-source/scheme-portfolios/all-schemes-monthly-portfolio---as-on-31st-may-2026.xlsx?sfvrsn=1e792ce4_2
Monthly Portfolio Disclosure - April 2026 -> https://www.sbimf.com/docs/default-source/scheme-portfolios/all-schemes-monthly-portfolio---as-on-30th-april-2026.xlsx?sfvrsn=fbc4d18a_2
Monthly Portfolio Disclosure - March 2026 -> https://www.sbimf.com/docs/default-source/scheme-portfolios/all-schemes-monthly-portfolio---as-on-31st-march-2026.xlsx?sfvrsn=e7c44034_2
Monthly Portfolio Disclosure - February 2026 -> https://www.sbimf.com/docs/default-source/scheme-portfolios/all-schemes-monthly-portfolio---as-on-28th-february-2026.xlsx?sfvrsn=5ca2382d_2
Monthly Portfolio Disclosure - January 2026 -> https://www.sbimf.com/docs/default-source/scheme-portfolios/all-schemes-monthly-portfolio---as-on-31st-january-2026.xlsx?sfvrsn=bf862a46_2


In [6]:
import requests

url = "https://www.sbimf.com/docs/default-source/scheme-portfolios/all-schemes-monthly-portfolio---as-on-31st-march-2026.xlsx?sfvrsn=e7c44034_2"
headers = {"User-Agent": "Mozilla/5.0"}

resp = requests.get(url, headers=headers, timeout=30)
print("Status:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))
print("Size (bytes):", len(resp.content))

with open("/content/sbi_march2026.xlsx", "wb") as f:
    f.write(resp.content)

Status: 200
Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
Size (bytes): 1946219


In [7]:
import openpyxl

wb = openpyxl.load_workbook("/content/sbi_march2026.xlsx", read_only=True, data_only=True)
print("Number of sheets:", len(wb.sheetnames))
print("Sheet names:", wb.sheetnames[:20])  # first 20 in case there are many

# Peek at the first sheet's first 15 rows
ws = wb[wb.sheetnames[0]]
for i, row in enumerate(ws.iter_rows(values_only=True)):
    print(row)
    if i > 15:
        break

Number of sheets: 129
Sheet names: ['Index', 'SMEEF', 'SLMF', 'SLTEF', 'SMGLF', 'SEHF', 'SMIF', 'SCOF', 'STOF', 'SHOF', 'SCF', 'SNIF', 'SMCBF-SP', 'SOF', 'SMMDF', 'SLF', 'SDBF', 'SSF', 'SCRF', 'SFEF']
('Index', None, None)
(None, None, None)
('Scheme Code', 'Scheme Short code', 'Scheme Name')
('007', 'SMEEF', 'SBI ESG Exclusionary Strategy Fund')
('017', 'SLMF', 'SBI Large and Midcap Fund')
('018', 'SLTEF', 'SBI ELSS Tax Saver Fund')
('021', 'SMGLF', 'SBI MNC Fund')
('024', 'SEHF', 'SBI Equity Hybrid Fund')
('028', 'SMIF', 'SBI Medium to Long Duration Fund (Erstwhile known as SBI Magnum Income Fund)')
('033', 'SCOF', 'SBI Consumption Opportunities Fund')
('034', 'STOF', 'SBI Technology Opportunities Fund')
('035', 'SHOF', 'SBI Healthcare Opportunities Fund')
('036', 'SCF', 'SBI Contra Fund')
('055', 'SNIF', 'SBI Nifty Index Fund')
('056', 'SMCBF-SP', "SBI Children's Fund - Savings Plan (Erstwhile known as SBI Magnum Children's Benefit Fund-Svg P)")
('057', 'SOF', 'SBI Overnight Fund')


In [8]:
# Step 1: read the full Index sheet and search for your target category names
ws_index = wb["Index"]
index_rows = list(ws_index.iter_rows(values_only=True))

# print every row containing "Cap" so we can see all fund name variants
for row in index_rows:
    if row and row[2] and "Cap" in str(row[2]):
        print(row)

('103', 'SBLUECHIP', 'SBI Large Cap Fund')
('652', 'SMCF', 'SBI MultiCap Fund')


In [9]:
# Step 2: once you spot the short code for e.g. "SBI Large Cap Fund" or "SBI Small Cap Fund",
# swap it in below and peek at the actual holdings structure
target_sheet = "SBLUECHIP"

ws_target = wb[target_sheet]
for i, row in enumerate(ws_target.iter_rows(values_only=True)):
    print(row)
    if i > 20:
        break

(None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, Non

In [10]:
import pandas as pd
import datetime

def parse_scheme_sheet(wb, sheet_name):
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))

    scheme_name = None
    statement_date = None
    holdings = []

    for row in rows:
        if row[2] == "SCHEME NAME :":
            scheme_name = row[3]
        elif row[2] == "PORTFOLIO STATEMENT AS ON :":
            statement_date = row[3]
        elif (
            row[1] not in (None, "")
            and row[2] not in (None, "")
            and isinstance(row[7], (int, float))          # % to AUM must be numeric
            and isinstance(row[3], str) and row[3].startswith("IN")  # real ISIN
        ):
            holdings.append({
                "stock_code": row[1],
                "instrument_name": row[2],
                "isin": row[3],
                "sector": row[4],
                "quantity": row[5],
                "market_value_lakhs": row[6],
                "pct_to_aum": row[7],
            })

    df = pd.DataFrame(holdings)
    df["scheme_name"] = scheme_name
    df["statement_date"] = statement_date
    return df

# Test it
df_test = parse_scheme_sheet(wb, "SBLUECHIP")
print("Scheme:", df_test["scheme_name"].iloc[0])
print("Statement date:", df_test["statement_date"].iloc[0])
print("Number of holdings:", len(df_test))
print("Sum of % to AUM (equity only):", df_test["pct_to_aum"].sum())
df_test.head(10)

Scheme: SBI Large Cap Fund
Statement date: 2026-03-31 00:00:00
Number of holdings: 47
Sum of % to AUM (equity only): 97.33000000000001


,stock_code,instrument_name,isin,sector,quantity,market_value_lakhs,pct_to_aum,scheme_name,statement_date
0,100006,HDFC Bank Ltd.,INE040A01034,Banks,60500000,442587.75,9.05,SBI Large Cap Fund,2026-03-31
1,100012,ICICI Bank Ltd.,INE090A01021,Banks,32850000,396138.15,8.10,SBI Large Cap Fund,2026-03-31
2,100002,Reliance Industries Ltd.,INE002A01018,Petroleum Products,24500000,329255.50,6.73,SBI Large Cap Fund,2026-03-31
3,100005,Larsen & Toubro Ltd.,INE018A01030,Construction,7400000,259303.40,5.30,SBI Large Cap Fund,2026-03-31
4,100010,State Bank of India,INE062A01020,Banks,19106000,187124.16,3.82,SBI Large Cap Fund,2026-03-31
5,100173,Asian Paints Ltd.,INE021A01026,Consumer Durables,8300000,179711.60,3.67,SBI Large Cap Fund,2026-03-31
6,100706,HDFC Life Insurance Company Ltd.,INE795G01014,Insurance,29000000,171274.00,3.50,SBI Large Cap Fund,2026-03-31
7,100003,Infosys Ltd.,INE009A01021,IT - Software,13417914,167804.43,3.43,SBI Large Cap Fund,2026-03-31
8,100104,Kotak Mahindra Bank Ltd.,INE237A01036,Banks,46000000,162564.00,3.32,SBI Large Cap Fund,2026-03-31
9,100024,Axis Bank Ltd.,INE238A01034,Banks,13750000,159678.75,3.26,SBI Large Cap Fund,2026-03-31


In [11]:
import requests
from bs4 import BeautifulSoup
import openpyxl

amc_slugs = {
    "SBI": "SBI-Mutual-Fund",
    "HDFC": "HDFC-Mutual-Fund",
    "ICICI": "ICICI-Prudential-Mutual-Fund",
    "Kotak": "Kotak-Mahindra-Mutual-Fund",
    "Nippon": "Nippon-India-Mutual-Fund",
}
headers = {"User-Agent": "Mozilla/5.0"}
all_links = {}

for name, slug in amc_slugs.items():
    url = f"https://www.advisorkhoj.com/form-download-centre/Mutual/{slug}"
    resp = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")
    links = {}
    for a in soup.find_all("a", href=True):
        text = a.get_text(strip=True)
        href = a["href"]
        if "Monthly Portfolio Disclosure" in text and ".xlsx" in href:
            links[text] = href
    all_links[name] = links
    print(name, ":", len(links), "files found")

SBI : 41 files found
HDFC : 0 files found
ICICI : 0 files found
Kotak : 6 files found
Nippon : 0 files found


In [12]:
amc_slugs = {
    "SBI": "SBI-Mutual-Fund",
    "HDFC": "HDFC-Mutual-Fund",
    "ICICI": "ICICI-Prudential-Mutual-Fund",
    "Kotak": "Kotak-Mahindra-Mutual-Fund",
    "Nippon": "Nippon-India-Mutual-Fund",
}
headers = {"User-Agent": "Mozilla/5.0"}
all_links = {}

for name, slug in amc_slugs.items():
    url = f"https://www.advisorkhoj.com/form-download-centre/Mutual/{slug}/Monthly-Portfolio-Disclosures"
    resp = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")
    links = {}
    for a in soup.find_all("a", href=True):
        text = a.get_text(strip=True)
        href = a["href"]
        if "Monthly Portfolio Disclosure" in text and ".xlsx" in href:
            links[text] = href
    all_links[name] = links
    print(name, ":", len(links), "files found")

SBI : 41 files found
HDFC : 0 files found
ICICI : 0 files found
Kotak : 6 files found
Nippon : 0 files found


In [13]:
import time

for name, slug in amc_slugs.items():
    url = f"https://www.advisorkhoj.com/form-download-centre/Mutual/{slug}/Monthly-Portfolio-Disclosures"
    resp = requests.get(url, headers=headers, timeout=15)
    print(name, "-> status:", resp.status_code, "| length:", len(resp.text))
    time.sleep(3)  # pause between requests

SBI -> status: 200 | length: 170471
HDFC -> status: 200 | length: 171938
ICICI -> status: 200 | length: 162950
Kotak -> status: 200 | length: 163922
Nippon -> status: 200 | length: 164463


In [14]:
url = "https://www.advisorkhoj.com/form-download-centre/Mutual/HDFC-Mutual-Fund/Monthly-Portfolio-Disclosures"
resp = requests.get(url, headers=headers, timeout=15)
soup = BeautifulSoup(resp.text, "html.parser")

for a in soup.find_all("a", href=True):
    if "Portfolio" in a.get_text():
        print(repr(a.get_text(strip=True)), "->", a["href"])

'Mutual Fund Portfolio Overlap' -> /mutual-funds-research/mutual-fund-portfolio-overlap
'HDFC  Monthly Portfolio Disclosures' -> #
'Monthly Portfolio Disclosure - May 2026' -> https://www.hdfcfund.com/statutory-disclosure/portfolio/monthly-portfolio
'Monthly Portfolio Disclosure - April 2026' -> https://www.hdfcfund.com/statutory-disclosure/portfolio/monthly-portfolio
'Monthly Portfolio Disclosure - March 2026' -> https://www.hdfcfund.com/statutory-disclosure/portfolio/monthly-portfolio
'Monthly Portfolio Disclosure - February 2026' -> https://www.hdfcfund.com/statutory-disclosure/portfolio/monthly-portfolio
'Monthly Portfolio Disclosure - January 2026' -> https://www.hdfcfund.com/statutory-disclosure/portfolio/monthly-portfolio
'Monthly Portfolio Disclosure - December 2025' -> https://www.hdfcfund.com/statutory-disclosure/portfolio/monthly-portfolio
'Monthly Portfolio Disclosure - November 2025' -> https://www.hdfcfund.com/statutory-disclosure/portfolio/monthly-portfolio
'Monthly Port

In [15]:
url = "https://www.advisorkhoj.com/form-download-centre/Mutual/Kotak-Mahindra-Mutual-Fund/Monthly-Portfolio-Disclosures"
resp = requests.get(url, headers=headers, timeout=15)
soup = BeautifulSoup(resp.text, "html.parser")

for a in soup.find_all("a", href=True):
    if "Portfolio" in a.get_text():
        print(repr(a.get_text(strip=True)), "->", a["href"])

'Mutual Fund Portfolio Overlap' -> /mutual-funds-research/mutual-fund-portfolio-overlap
'Kotak Mahindra  Monthly Portfolio Disclosures' -> #
'Monthly Portfolio Disclosure - May 2026' -> https://vatseelabs-s3.kotakmf.com/FormsDownloads/Portfolios/Consolidated-SEBI-Portfolio-as-on-May-31,-2026/ConsolidatedSEBIPortfolioMay2026.xlsx
'Monthly Portfolio Disclosure - April 2026' -> https://vatseelabs-s3.kotakmf.com/FormsDownloads/Portfolios/Consolidated-Portfolio-as-on-April-30,-2026/ConsolidatedSEBIPortfolioApril2026.xlsx
'Monthly Portfolio Disclosure - March 2026' -> https://vatseelabs-s3.kotakmf.com/FormsDownloads/Portfolios/Consolidated-Portfolio-as-on-March-31,-2026/ConsolidatedSEBIPortfolioMarch2026.xlsx
'Monthly Portfolio Disclosure - February 2026' -> https://vatseelabs-s3.kotakmf.com/FormsDownloads/Portfolios/Consolidated-Portfolio-as-on-February-28,-2026/ConsolidatedSEBIPortfolioFebruary2026.xlsx
'Monthly Portfolio Disclosure - January 2026' -> https://vatseelabs-s3.kotakmf.com/Form

In [16]:
def get_portfolio_links(slug):
    url = f"https://www.advisorkhoj.com/form-download-centre/Mutual/{slug}/Monthly-Portfolio-Disclosures"
    resp = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")
    links = {}
    for a in soup.find_all("a", href=True):
        text = a.get_text(strip=True)
        href = a["href"]
        if "Monthly Portfolio Disclosure" in text and (href.endswith(".xls") or href.endswith(".xlsx")):
            links[text] = href
    return links

kotak_links = get_portfolio_links("Kotak-Mahindra-Mutual-Fund")
print(len(kotak_links), "files found")

41 files found


In [17]:
import requests
import pandas as pd

# Use the most recent file, which is .xlsx (easiest to start with)
url = "https://vatseelabs-s3.kotakmf.com/FormsDownloads/Portfolios/Consolidated-Portfolio-as-on-March-31,-2026/ConsolidatedSEBIPortfolioMarch2026.xlsx"
resp = requests.get(url, headers=headers, timeout=30)
print("Status:", resp.status_code)
print("Size:", len(resp.content))

with open("/content/kotak_march2026.xlsx", "wb") as f:
    f.write(resp.content)

Status: 200
Size: 3831629


In [18]:
import openpyxl

wb_kotak = openpyxl.load_workbook("/content/kotak_march2026.xlsx", read_only=True, data_only=True)
print("Number of sheets:", len(wb_kotak.sheetnames))
print("Sheet names:", wb_kotak.sheetnames[:20])

# Peek at first sheet
ws = wb_kotak[wb_kotak.sheetnames[0]]
for i, row in enumerate(ws.iter_rows(values_only=True)):
    print(row)
    if i > 15:
        break

Number of sheets: 121
Sheet names: ['V3I', 'TIF', 'TCH', 'TAL', 'STF', 'SRF', 'SPO', 'SOF', 'SIL', 'SIF', 'SEF', 'ROF', 'QOF', 'Q3T', 'Q3I', 'OVR', 'NVF', 'NTF', 'NSI', 'NNF']
(None, None, 'Portfolio of Kotak Nifty200 Value 30 Index Fund as on 31-Mar-2026', None, None, None, None, None, None)
('Name of Instrument', None, None, 'ISIN Code', 'Industry', 'Yield', 'Quantity', 'Market Value (Rs.in Lacs)', '% to Net Assets')
('Equity & Equity related', None, None, None, None, None, None, None, None)
(None, 'Listed/Awaiting listing on Stock Exchange', None, None, None, None, None, None, None)
(None, ' ', 'Oil And Natural Gas Corporation Ltd.', 'INE213A01029', 'Oil', None, 24876, 70.81, 6.24)
(None, ' ', 'NTPC LTD', 'INE733E01010', 'Power', None, 18002, 66.72, 5.880000000000001)
(None, ' ', 'Coal India Limited', 'INE522F01014', 'Consumable Fuels', None, 14507, 65.35, 5.760000000000001)
(None, ' ', 'Tata Steel Ltd.', 'INE081A01020', 'Ferrous Metals', None, 33304, 63.9, 5.63)
(None, ' ', 'VEDANT

In [19]:
import re

def build_kotak_scheme_map(wb):
    scheme_map = {}
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        for i, row in enumerate(ws.iter_rows(values_only=True)):
            if i > 3:  # title is always in the first few rows
                break
            for cell in row:
                if cell and isinstance(cell, str) and "Portfolio of" in cell:
                    match = re.search(r"Portfolio of (.+?) as on", cell)
                    if match:
                        scheme_map[sheet_name] = match.group(1).strip()
    return scheme_map

kotak_scheme_map = build_kotak_scheme_map(wb_kotak)

# Show every "Cap" fund
for code, name in kotak_scheme_map.items():
    if "Cap" in name:
        print(code, "->", name)

MID -> Kotak Small Cap Fund
K30 -> Kotak Large Cap Fund


In [20]:
for code, name in kotak_scheme_map.items():
    if "cap" in name.lower() or "emerging" in name.lower() or "flexi" in name.lower() or "multi" in name.lower():
        print(code, "->", name)

SIF -> Kotak Nifty Smallcap 50 Index Fund
SEF -> Kotak Flexicap Fund
NSI -> Kotak Nifty Smallcap 250 Index Fund
NMX -> Kotak Nifty Midcap 150 Index Fund
NMT -> Kotak Nifty Midcap 150 ETF
MUC -> Kotak Multicap Fund
MTF -> Kotak Nifty Midcap 50 ETF
MMI -> Kotak Nifty Midcap 150 Momentum 50 Index Fund
MID -> Kotak Small Cap Fund
MFF -> Kotak Multi Factor Passive FOF
MCF -> Kotak Nifty Midcap 50 Index Fund
MAF -> Kotak Multi Asset Allocation Fund
KOP -> Kotak Large & Midcap Fund
K30 -> Kotak Large Cap Fund
GEM -> Kotak Global Emerging Market Overseas Equity Omni FOF
FOF -> Kotak Multi Asset Omni FOF
EME -> Kotak Midcap Fund


In [21]:
# Rebuild SBI's scheme map the same way, then search wider
sbi_scheme_map = {}
for row in ws_index.iter_rows(values_only=True):  # reuse the SBI Index sheet from earlier
    if row and row[1] and row[2]:
        sbi_scheme_map[row[1]] = row[2]

for code, name in sbi_scheme_map.items():
    if "cap" in name.lower() or "emerging" in name.lower() or "flexi" in name.lower() or "multi" in name.lower():
        print(code, "->", name)

SLMF -> SBI Large and Midcap Fund
SMIDCAP -> SBI Midcap Fund
SFLEXI -> SBI Flexicap Fund
SMAAF -> SBI Multi Asset Allocation Fund
SBLUECHIP -> SBI Large Cap Fund
SSCF -> SBI Smallcap Fund
SMCF -> SBI MultiCap Fund
SNM150IF -> SBI Nifty Midcap 150 Index Fund
SNS250IF -> SBI Nifty Smallcap 250 Index Fund
SNM150M50ETF -> SBI Nifty Midcap 150 Momentum 50 ETF
SNM150ETF -> SBI Nifty Midcap150 ETF


In [22]:
!pip install xlrd

In [ ]:
import pandas as pd
import re
import time

def load_workbook_df(path):
    if path.endswith(".xls"):
        return pd.read_excel(path, sheet_name=None, header=None, engine="xlrd")
    else:
        return pd.read_excel(path, sheet_name=None, header=None, engine="openpyxl")

def parse_sbi_scheme(df, scheme_name):
    holdings = []
    statement_date = None
    for _, row in df.iterrows():
        row = row.tolist()
        if len(row) < 8:
            continue
        if row[2] == "PORTFOLIO STATEMENT AS ON :":
            statement_date = row[3]
        elif isinstance(row[3], str) and row[3].startswith("IN") and isinstance(row[7], (int, float)):
            holdings.append({
                "instrument_name": row[2], "isin": row[3], "sector": row[4],
                "quantity": row[5], "market_value_lakhs": row[6], "pct_to_aum": row[7],
            })
    out = pd.DataFrame(holdings)
    out["scheme_name"] = scheme_name
    out["statement_date"] = statement_date
    out["amc"] = "SBI"
    return out

def parse_kotak_scheme(df, scheme_name):
    holdings = []
    statement_date = None
    for _, row in df.head(3).iterrows():
        for cell in row:
            if isinstance(cell, str) and "as on" in cell:
                m = re.search(r"as on ([\d\-A-Za-z]+)", cell)
                if m:
                    statement_date = m.group(1)
    for _, row in df.iterrows():
        row = row.tolist()
        if len(row) < 9:
            continue
        if isinstance(row[3], str) and row[3].startswith("IN") and isinstance(row[8], (int, float)):
            holdings.append({
                "instrument_name": row[2], "isin": row[3], "sector": row[4],
                "quantity": row[6], "market_value_lakhs": row[7], "pct_to_aum": row[8],
            })
    out = pd.DataFrame(holdings)
    out["scheme_name"] = scheme_name
    out["statement_date"] = statement_date
    out["amc"] = "Kotak"
    return out

sbi_targets = {
    "SBLUECHIP": "SBI Large Cap Fund", "SMIDCAP": "SBI Midcap Fund",
    "SSCF": "SBI Smallcap Fund", "SLMF": "SBI Large and Midcap Fund",
    "SMCF": "SBI MultiCap Fund", "SFLEXI": "SBI Flexicap Fund",
}
kotak_targets = {
    "K30": "Kotak Large Cap Fund", "EME": "Kotak Midcap Fund",
    "MID": "Kotak Small Cap Fund", "KOP": "Kotak Large & Midcap Fund",
    "MUC": "Kotak Multicap Fund", "SEF": "Kotak Flexicap Fund",
}

all_holdings = []
errors = []

def run_amc(links, targets, parser, amc_name):
    for label, url in links.items():
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            ext = ".xlsx" if url.lower().endswith(".xlsx") else ".xls"
            path = f"/content/tmp_{amc_name}{ext}"
            with open(path, "wb") as f:
                f.write(resp.content)
            sheets = load_workbook_df(path)
            for code, scheme_name in targets.items():
                if code in sheets:
                    d = parser(sheets[code], scheme_name)
                    d["file_label"] = label
                    all_holdings.append(d)
                else:
                    errors.append(f"{amc_name} {label}: sheet {code} not found")
        except Exception as e:
            errors.append(f"{amc_name} {label}: {e}")
        time.sleep(1)

run_amc(sbi_links, sbi_targets, parse_sbi_scheme, "SBI")
run_amc(kotak_links, kotak_targets, parse_kotak_scheme, "Kotak")

holdings_master = pd.concat(all_holdings, ignore_index=True)
print("Total rows:", holdings_master.shape[0])
print("\nRows per scheme:")
print(holdings_master["scheme_name"].value_counts())
print("\nErrors:", len(errors))
for e in errors[:20]:
    print(" -", e)

holdings_master.to_csv("/content/holdings_master.csv", index=False)
print("\nSaved to /content/holdings_master.csv")

NameError: name 'sbi_links' is not defined

In [23]:
sbi_links = get_portfolio_links("SBI-Mutual-Fund")
kotak_links = get_portfolio_links("Kotak-Mahindra-Mutual-Fund")

print("SBI:", len(sbi_links), "files")
print("Kotak:", len(kotak_links), "files")

SBI: 0 files
Kotak: 41 files


In [24]:
url = "https://www.advisorkhoj.com/form-download-centre/Mutual/SBI-Mutual-Fund/Monthly-Portfolio-Disclosures"
resp = requests.get(url, headers=headers, timeout=15)
print("Status:", resp.status_code)
print("Length:", len(resp.text))
print("Contains 'Monthly Portfolio Disclosure'?", "Monthly Portfolio Disclosure" in resp.text)
print("Contains '.xls'?", ".xls" in resp.text)

Status: 200
Length: 170344
Contains 'Monthly Portfolio Disclosure'? True
Contains '.xls'? True


In [25]:
def get_portfolio_links(slug):
    url = f"https://www.advisorkhoj.com/form-download-centre/Mutual/{slug}/Monthly-Portfolio-Disclosures"
    resp = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")
    links = {}
    for a in soup.find_all("a", href=True):
        text = a.get_text(strip=True)
        href = a["href"]
        if "Monthly Portfolio Disclosure" in text and (".xls" in href or ".xlsx" in href):
            links[text] = href
    return links

sbi_links = get_portfolio_links("SBI-Mutual-Fund")
kotak_links = get_portfolio_links("Kotak-Mahindra-Mutual-Fund")
print("SBI:", len(sbi_links), "files")
print("Kotak:", len(kotak_links), "files")

SBI: 41 files
Kotak: 41 files


In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
sbi_targets = {
    "SBLUECHIP": "SBI Large Cap Fund", "SMIDCAP": "SBI Midcap Fund",
    "SSCF": "SBI Smallcap Fund", "SLMF": "SBI Large and Midcap Fund",
    "SMCF": "SBI MultiCap Fund", "SFLEXI": "SBI Flexicap Fund",
}
kotak_targets = {
    "K30": "Kotak Large Cap Fund", "EME": "Kotak Midcap Fund",
    "MID": "Kotak Small Cap Fund", "KOP": "Kotak Large & Midcap Fund",
    "MUC": "Kotak Multicap Fund", "SEF": "Kotak Flexicap Fund",
}

all_holdings = []
errors = []

def run_amc(links, targets, parser, amc_name):
    for label, url in links.items():
        try:
            resp = requests.get(url, headers=headers, timeout=30)
            ext = ".xlsx" if url.lower().split("?")[0].endswith(".xlsx") else ".xls"
            path = f"/content/tmp_{amc_name}{ext}"
            with open(path, "wb") as f:
                f.write(resp.content)
            sheets = load_workbook_df(path)
            for code, scheme_name in targets.items():
                if code in sheets:
                    d = parser(sheets[code], scheme_name)
                    d["file_label"] = label
                    all_holdings.append(d)
                else:
                    errors.append(f"{amc_name} {label}: sheet {code} not found")
        except Exception as e:
            errors.append(f"{amc_name} {label}: {e}")
        time.sleep(1)

run_amc(sbi_links, sbi_targets, parse_sbi_scheme, "SBI")
run_amc(kotak_links, kotak_targets, parse_kotak_scheme, "Kotak")

holdings_master = pd.concat(all_holdings, ignore_index=True)
print("Total rows:", holdings_master.shape[0])
print("\nRows per scheme:")
print(holdings_master["scheme_name"].value_counts())
print("\nErrors:", len(errors))
for e in errors[:20]:
    print(" -", e)

save_path = "/content/drive/MyDrive/StyleDrift_Project/processed/holdings_master.csv"
holdings_master.to_csv(save_path, index=False)
print("\nSaved to", save_path)

NameError: name 'parse_sbi_scheme' is not defined